#### 1. Import libraries.

In [1]:
%cd ..

/Users/mateuszgrzyb/Projekty/algorithmic_trading


In [2]:
import joblib

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import r2_score

from src.utils.tools import calculate_financial_ratios

TARGET = 'label_2000_100_252_pct_change'

#### 2. Load data.

In [3]:
tr = pd.read_feather('data/abt_clean/tr.feather')
va = pd.read_feather('data/abt_clean/va.feather')
te = pd.read_feather('data/abt_clean/te.feather')

In [5]:
features_to_model = ['graham_number_vs_price', 
                     'eps',
                     'price_to_sales', 
                     'book_value_per_share',
                     'price_to_earnings',
                     'market_cap',
                     'price_to_book']

#### 3. Load model.

In [4]:
model = joblib.load('models/rf_20260506_0103.joblib')

#### 4. Score data.

In [6]:
pred_tr = model.predict(tr[features_to_model])
pred_va = model.predict(va[features_to_model])
pred_te = model.predict(te[features_to_model])

tr['pred'] = pred_tr
va['pred'] = pred_va
te['pred'] = pred_te

#### 7. Search for optimal "n".

##### 7.1. Validation dataset.

In [14]:
results = []
for n in range(1, 21):
    cohort_returns = []
    for quarter in va.date.unique():
        cohort_return = va[va.date == quarter]\
            .sort_values('pred', ascending=False)\
            .iloc[0:n][TARGET]\
            .mean()
        cohort_returns.append(cohort_return)

    # Agregacja metryk dla danego N
    avg_return = np.mean(cohort_returns)
    volatility = np.std(cohort_returns)
    worst_cohort = np.min(cohort_returns)
    sharpe_proxy = avg_return / volatility if volatility > 0 else 0
    
    results.append({
        'N': n,
        'Avg_Return': avg_return,
        'Volatility': volatility,
        'Worst_Drawdown': worst_cohort,
        'Sharpe': sharpe_proxy
    })
results_va = pd.DataFrame(results)

In [15]:
results_va.head(10)

,N,Avg_Return,Volatility,Worst_Drawdown,Sharpe
0,1,24.619686,40.229698,-53.888889,0.611978
1,2,25.416480,29.966630,-19.262851,0.848159
2,3,26.291312,30.627926,-15.449003,0.858410
3,4,22.645665,30.229102,-25.953564,0.749135
4,5,22.144197,34.852179,-25.653871,0.635375
5,6,20.638257,31.774467,-25.528100,0.649523
6,7,19.388756,30.305988,-24.332803,0.639766
7,8,20.057594,28.274954,-21.660409,0.709377
8,9,19.626028,27.670464,-21.034271,0.709277
9,10,19.229137,26.221051,-18.371228,0.733347


##### 7.2. Test dataset.

In [16]:
results = []
for n in range(1, 21):
    cohort_returns = []
    for quarter in te.date.unique():
        cohort_return = te[te.date == quarter]\
            .sort_values('pred', ascending=False)\
            .iloc[0:n][TARGET]\
            .mean()
        cohort_returns.append(cohort_return)

    # Agregacja metryk dla danego N
    avg_return = np.mean(cohort_returns)
    volatility = np.std(cohort_returns)
    worst_cohort = np.min(cohort_returns)
    sharpe_proxy = avg_return / volatility if volatility > 0 else 0
    
    results.append({
        'N': n,
        'Avg_Return': avg_return,
        'Volatility': volatility,
        'Worst_Drawdown': worst_cohort,
        'Sharpe': sharpe_proxy
    })
results_te = pd.DataFrame(results)

In [17]:
results_te.head(10)

,N,Avg_Return,Volatility,Worst_Drawdown,Sharpe
0,1,27.562553,76.748124,-41.082965,0.359130
1,2,19.611791,51.050773,-25.845797,0.384162
2,3,19.773557,38.959221,-26.454333,0.507545
3,4,17.517679,31.640068,-23.812648,0.553655
4,5,17.348085,27.757294,-24.251492,0.624992
5,6,14.434048,27.583089,-28.714446,0.523293
6,7,13.033385,24.495819,-30.864532,0.532066
7,8,11.222168,21.531762,-26.776598,0.521191
8,9,10.548409,21.703301,-28.064750,0.486028
9,10,9.892388,20.561791,-22.986833,0.481105


##### 7.3. Both.

In [18]:
va_te = pd.concat([va, te])
results = []
for n in range(1, 21):
    cohort_returns = []
    for quarter in va_te.date.unique():
        cohort_return = va_te[va_te.date == quarter]\
            .sort_values('pred', ascending=False)\
            .iloc[0:n]['label_2000_100_252_pct_change']\
            .mean()
        cohort_returns.append(cohort_return)

    # Agregacja metryk dla danego N
    avg_return = np.mean(cohort_returns)
    volatility = np.std(cohort_returns)
    worst_cohort = np.min(cohort_returns)
    sharpe_proxy = avg_return / volatility if volatility > 0 else 0
    
    results.append({
        'N': n,
        'Avg_Return': avg_return,
        'Volatility': volatility,
        'Worst_Drawdown': worst_cohort,
        'Sharpe': sharpe_proxy
    })
results_all = pd.DataFrame(results)

In [19]:
results_all.head(10)  # n=3 seems to work best for all samples

,N,Avg_Return,Volatility,Worst_Drawdown,Sharpe
0,1,25.208260,49.741165,-53.888889,0.506789
1,2,24.255542,35.284933,-25.845797,0.687419
2,3,24.987761,32.570187,-26.454333,0.767197
3,4,21.620068,30.585374,-25.953564,0.706876
4,5,21.184975,33.608234,-25.653871,0.630351
5,6,19.397415,31.080822,-28.714446,0.624096
6,7,18.117682,29.346787,-30.864532,0.617365
7,8,18.290509,27.290880,-26.776598,0.670206
8,9,17.810504,26.831229,-28.064750,0.663798
9,10,17.361787,25.466453,-22.986833,0.681751
